# Inside the Model: Logistic Regression

## Correct labels are only part of the story

**30–35 minutes**

Two models give the same predicted labels. One is more confident. Is that always an advantage? Follow the probability assigned to each observed label to find out.

### Learning objectives

- distinguish a predicted probability from the label obtained by thresholding it;
- explain how log loss rewards confidence in the observed label;
- compare how one changed observed label affects two models' losses.

Predict first, explain with a partner, and only then verify with Python. The optional challenge is additional.


### Before you start

**Prerequisite:** Logistic Regression and Log Loss. The optional challenge also uses the log-loss gradient.

**Recall:** the sigmoid gives a probability $p$ of label 1. The probability of label 0 is $1-p$. A threshold converts probabilities into decisions; log loss instead asks how much probability the model assigned to the **observed** label.

For one observation, the loss is

$$\ell=-\ln(p)\quad\text{if }y=1,\qquad
\ell=-\ln(1-p)\quad\text{if }y=0.$$

Use natural logarithms. A larger probability for the observed label means a smaller loss.


### Four observations, two fixed models

The probability columns are supplied so that the task focuses on the mechanism, not on exponential arithmetic.

| Observation | Input $x$ | Original label $y$ | Model A: probability of label 1 | Model B: probability of label 1 |
|--:|--:|--:|--:|--:|
| 1 | −1 | 0 | 0.25 | 0.10 |
| 2 | −1 | 0 | 0.25 | 0.10 |
| 3 | 1 | 1 | 0.75 | 0.90 |
| 4 | 1 | 1 | 0.75 | 0.90 |

Count each row once with equal weight, including the repeated inputs. Both models have zero intercept: A uses $p=\sigma((\ln3)x)$ and B uses $p=\sigma((\ln9)x)$. There is no regularisation.

Predict label 1 when $p\geq0.5$, otherwise label 0. The models are **fixed candidates**: we compare their objectives without refitting them.

---

## 1. Predict the comparison

**About 4 minutes · No Python**

Which labels will A and B predict? Which is more confident in those decisions? Explain whether counting correct labels can distinguish them on the original data.

Before calculating, predict which model has the smaller mean log loss. State the probability that matters to your reasoning: probability of label 1, or probability of the observed label?

**Your prediction and justification:** _Write here._


## 2. Follow the observed label

**About 8 minutes · No Python**

**Worked example:** observation 1 has label 0. Model A assigns that observed label probability $1-0.25=0.75$, so its loss is $-\ln(0.75)\approx0.288$. Using $-\ln(0.25)$ here would score the wrong label.

Complete the table. You can reuse values when the same probability occurs; no repeated arithmetic is needed.

| Observation | A: probability of observed label | B: probability of observed label | A: loss | B: loss |
|--:|--:|--:|--:|--:|
| 1 | 0.75 | … | 0.288 | … |
| 2 | … | … | … | … |
| 3 | … | … | … | … |
| 4 | … | … | … | … |

Average the four losses for each model. You may use $-\ln(0.90)\approx0.105$ or a basic calculator. Explain why two models with identical labels can still have different training objectives.

**Your calculation and explanation:** _Write here._

<details>
<summary>Hint</summary>

First identify the observed label. Use $p$ for label 1 and $1-p$ for label 0.

</details>


## 3. Change one observed label

**About 8 minutes · No Python**

Change only **observation 4's label from 1 to 0**. Its input stays at $x=1$. Keep both models and the threshold unchanged.

Predict again before calculating:

- Will either model's predicted labels or probabilities change?
- Will the models still have the same number of correct decisions?
- Which model do you expect to have lower loss now, and why?

**Your new prediction:** _Write here._

Only one row's loss changes. For observation 4, calculate the probability now assigned to its observed label and replace the old contribution in the mean. Use $-\ln(0.25)\approx1.386$ and $-\ln(0.10)\approx2.303$ if helpful.

| Dataset | A: mean log loss | B: mean log loss | Preferred candidate |
|:--|--:|--:|:--|
| Original | … | … | … |
| Changed label | … | … | … |

Does the same number of mistakes imply the same cost? Explain the comparison using the changed row's probability, not just its correct/incorrect status.

**Your updated calculation and explanation:** _Write here._


---

## 4. Verify the probabilities and losses

**About 7 minutes · Begin after Sections 1–3**

Complete the two label expressions and the row-loss expression. In NumPy, `(p >= 0.5).astype(int)` converts probabilities into labels. The mean log-loss formula from the notes is

$$L=-\frac14\sum_i\left[y_i\ln p_i+(1-y_i)\ln(1-p_i)\right].$$

All supplied probabilities are strictly between 0 and 1, so `np.log` is safe here. The plot shows the individual contributions after the label change. We use the fixed probabilities directly; fitting a new estimator would change the experiment.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([-1, -1, 1, 1], dtype=float)
y_original = np.array([0, 0, 1, 1])
y_changed = y_original.copy()
y_changed[3] = 0
p_a = np.array([0.25, 0.25, 0.75, 0.75])
p_b = np.array([0.10, 0.10, 0.90, 0.90])


In [ ]:
# ADD YOUR CODE HERE -- replace the two None values.
labels_a = None
labels_b = None


In [ ]:
def mean_log_loss(y, p):
    # ADD YOUR CODE HERE -- replace None with the loss for every row.
    row_losses = None
    if row_losses is None:
        return None
    return np.mean(row_losses)


In [ ]:
if labels_a is None or labels_b is None or mean_log_loss(y_original, p_a) is None:
    print('Complete the label and loss expressions, then rerun this cell.')
else:
    print('Predicted labels A:', labels_a)
    print('Predicted labels B:', labels_b)
    print('Original mean loss A, B:', mean_log_loss(y_original, p_a), mean_log_loss(y_original, p_b))
    print('Changed mean loss A, B:', mean_log_loss(y_changed, p_a), mean_log_loss(y_changed, p_b))
    print('Correct after change A, B:', np.sum(labels_a == y_changed), np.sum(labels_b == y_changed))

    losses_a = -(y_changed * np.log(p_a) + (1 - y_changed) * np.log(1 - p_a))
    losses_b = -(y_changed * np.log(p_b) + (1 - y_changed) * np.log(1 - p_b))
    observations = np.arange(1, 5)
    plt.figure(figsize=(8, 4.5))
    plt.bar(observations - 0.18, losses_a, width=0.36, color='blue', label='Model A')
    plt.bar(observations + 0.18, losses_b, width=0.36, color='red', label='Model B')
    plt.xticks(observations, ['1', '2', '3', '4: changed label'])
    plt.ylabel('Loss contribution')
    plt.xlabel('Observation')
    plt.title('One confident mistake changes the comparison')
    plt.grid(True, axis='y', alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()


---

## 5. Explain the comparison

**About 5 minutes**

1. Why did changing the observed label affect the loss even though the fixed models' predictions did not change?
2. Relative to A, did B make an additional mistake or a more confident mistake? Support your answer with observation 4.
3. Does being more confident always help? State when it helped here and when it hurt. Would this four-row comparison establish which model predicts new data better?

**Your answers:** _Use one or two sentences for each._

## Optional Master's challenge

**A further 5 minutes · Use the gradient section of the deep-dive note**

For a zero-intercept model $p_i=\sigma(\beta x_i)$, the notes give

$$\frac{dL}{d\beta}=\frac14\sum_i(p_i-y_i)x_i.$$

Use the **changed** labels to evaluate this derivative at A and B. Which way would a small gradient-descent step move B's coefficient? Is A stationary? No fitting or proof of global optimality is required.

**Your calculation:** _Write here._

---

## Takeaway

Complete: **“A correct/incorrect label hides …, while log loss responds to …”**
